# Projeto Clustering — Pipeline final

Este notebook contém apenas a **modelagem de clustering patrimonial**.

De modo a combater a alta variação de valores a solução final usa uma estratégia **estratificada**. subdividindo os candidatos nos grupos a seguir

1. **Baixo patrimônio:** até **1 milhão de reais** 
2. **Patrimônio intermediário:** de **1 milhão a 10 milhões de reais** 
3. **Alto patrimônio:** de **10 milhões a 100 milhões de reais** 
4. **Ultra-alto patrimônio:** acima de **100 milhões de reais** 

O objetivo é gerar uma coluna final de cluster/perfil por candidato, mantendo a explicabilidade da tipologia.

## 1. Imports e configurações gerais

Altere os parâmetros desta seção caso queira testar novas métricas, pesos, cortes ou números de clusters.


In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

from sklearn.mixture import GaussianMixture
from scipy.spatial.distance import pdist, squareform

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [17]:
# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================

CAMINHO_DATASET = "features_patrimoniais_por_candidato.csv"
SEP = ";"

RANDOM_STATE = 42
N_INIT_KMEANS = 50

# Cortes patrimoniais 
CORTE_BAIXO = 1_000_000
CORTE_INTERMEDIARIO = 10_000_000
CORTE_ULTRA_ALTO = 100_000_000

# Número de clusters
K_BAIXO = 8
K_INTERMEDIARIO = 10
K_ALTO_REGULAR = 9
K_ULTRA = 7

# Clipping aplicado aos modelos de alto e ultra-alto patrimônio
CLIP_ALTO = (-5, 5)

# Se quiser recalcular tabelas de métricas exploratórias, mude para True.
# Por padrão fica False para o notebook rodar mais rápido.
RODAR_BUSCAS_METRICAS = False

# Intervalos de k usados se RODAR_BUSCAS_METRICAS=True
RANGES_K = {
    "baixo": range(4, 13),
    "intermediario": range(4, 13),
    "alto_regular": range(4, 14),
    "ultra": range(4, 13),
}


## 2. Carregar dataset de features por candidato

Este é o dataset usado nos experimentos de clustering. A unidade é o candidato (`SQ_CANDIDATO`).


In [18]:
features_candidatos = pd.read_csv(
    CAMINHO_DATASET,
    sep=SEP
)

print("Shape:", features_candidatos.shape)
features_candidatos.head()


Shape: (18219, 30)


,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,...,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,log_patrimonio_total,log_qtd_bens,SG_UF
0,10001595335,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,13.304687,1.098612,AC
1,10001595336,0,0,0,0,0,4000000,0,0,0,...,0.0,0.0,0.0,0.221638,3,2,0.654970,15.452369,1.386294,AC
2,10001595338,0,0,0,0,0,3000000,0,0,0,...,0.0,0.0,0.0,0.038462,2,2,0.926036,14.953344,1.098612,AC
3,10001595339,0,0,0,0,0,2500000,0,0,0,...,0.0,0.0,0.0,0.285714,2,2,0.591837,15.068274,1.098612,AC
4,10001595340,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,12.676079,1.098612,AC


In [19]:
# Conferências básicas
print("Candidatos únicos:", features_candidatos["SQ_CANDIDATO"].nunique())
print("Linhas:", len(features_candidatos))
print("Duplicados por SQ_CANDIDATO:", features_candidatos["SQ_CANDIDATO"].duplicated().sum())


Candidatos únicos: 18219
Linhas: 18219
Duplicados por SQ_CANDIDATO: 0


## 3. Definição das variáveis do modelo

Devido à grande variação de escala entre os valores patrimoniais dos candidatos, foi adotada uma transformação em escala logarítmica para as variáveis monetárias. Essa escolha reduz o efeito de valores extremos e permite que o modelo capture melhor diferenças relativas entre candidatos, sem que patrimônios muito elevados dominem completamente o processo de agrupamento.

As variáveis utilizadas no clustering combinam três dimensões complementares:

- **composição percentual por macro categoria**, para representar como o patrimônio está distribuído entre diferentes tipos de bens;
- **valores absolutos por macro categoria em escala logarítmica**, para preservar a informação de tamanho patrimonial sem distorcer o modelo por causa de outliers;
- **medidas de complexidade patrimonial**, para capturar a diversidade e a concentração dos bens declarados.


In [20]:
# Colunas percentuais por macro categoria
perc_cols = [
    "perc_ativos_financeiros",
    "perc_bens_luxo_colecao",
    "perc_creditos_direitos",
    "perc_dinheiro_especie",
    "perc_direitos_intangiveis",
    "perc_imoveis",
    "perc_outros",
    "perc_outros_atividade_profissional",
    "perc_participacoes_societarias",
    "perc_rural_agropecuario",
    "perc_veiculos"
]

# Valores absolutos por macro categoria
valor_cols = [
    "valor_ativos_financeiros",
    "valor_bens_luxo_colecao",
    "valor_creditos_direitos",
    "valor_dinheiro_especie",
    "valor_direitos_intangiveis",
    "valor_imoveis",
    "valor_outros",
    "valor_outros_atividade_profissional",
    "valor_participacoes_societarias",
    "valor_rural_agropecuario",
    "valor_veiculos"
]

# Criar logs de valores absolutos, se ainda não existirem
log_valor_cols = []

for col in valor_cols:
    nova_col = "log_" + col
    if nova_col not in features_candidatos.columns:
        features_candidatos[nova_col] = np.log1p(features_candidatos[col])
    log_valor_cols.append(nova_col)

complexidade_cols = [
    "log_qtd_bens",
    "qtd_macros_presentes",
    "indice_concentracao_macro"
]

features_modelo = perc_cols + log_valor_cols + complexidade_cols

# Conferir se todas as colunas existem
faltantes = [col for col in features_modelo if col not in features_candidatos.columns]
if faltantes:
    raise ValueError(f"Colunas faltantes no dataset: {faltantes}")

features_modelo


['perc_ativos_financeiros',
 'perc_bens_luxo_colecao',
 'perc_creditos_direitos',
 'perc_dinheiro_especie',
 'perc_direitos_intangiveis',
 'perc_imoveis',
 'perc_outros',
 'perc_outros_atividade_profissional',
 'perc_participacoes_societarias',
 'perc_rural_agropecuario',
 'perc_veiculos',
 'log_valor_ativos_financeiros',
 'log_valor_bens_luxo_colecao',
 'log_valor_creditos_direitos',
 'log_valor_dinheiro_especie',
 'log_valor_direitos_intangiveis',
 'log_valor_imoveis',
 'log_valor_outros',
 'log_valor_outros_atividade_profissional',
 'log_valor_participacoes_societarias',
 'log_valor_rural_agropecuario',
 'log_valor_veiculos',
 'log_qtd_bens',
 'qtd_macros_presentes',
 'indice_concentracao_macro']

## 4. Estratificação patrimonial

A clusterização final é feita dentro de faixas de patrimônio declarado. Isso evita comparar, no mesmo espaço de clustering, candidatos com escalas patrimoniais muito diferentes.


In [21]:
# ============================================================
# Estratos patrimoniais finais
# ============================================================

features_candidatos["estrato_base"] = pd.cut(
    features_candidatos["patrimonio_total"],
    bins=[0, CORTE_BAIXO, CORTE_INTERMEDIARIO, np.inf],
    labels=[
        "Baixo patrimônio",
        "Patrimônio intermediário",
        "Alto patrimônio"
    ],
    include_lowest=True
)

features_candidatos["estrato_base"].value_counts(dropna=False)


estrato_base
Patrimônio intermediário    7970
Alto patrimônio             5443
Baixo patrimônio            4806
Name: count, dtype: int64

In [22]:
# Separar alto patrimônio em alto regular e ultra-alto
mask_alto = features_candidatos["estrato_base"].eq("Alto patrimônio")

features_candidatos.loc[mask_alto, "subestrato_alto"] = pd.cut(
    features_candidatos.loc[mask_alto, "patrimonio_total"],
    bins=[CORTE_INTERMEDIARIO, CORTE_ULTRA_ALTO, np.inf],
    labels=[
        "Alto patrimônio",
        "Ultra-alto patrimônio"
    ],
    include_lowest=True
).astype("string")

features_candidatos["subestrato_alto"].value_counts(dropna=False)


subestrato_alto
<NA>                     12776
Alto patrimônio           4484
Ultra-alto patrimônio      959
Name: count, dtype: Int64

## 5. Pesos dos modelos finais

Os pesos foram definidos para preservar a composição patrimonial, mas reduzir a influência de categorias raras/residuais. Nos estratos de alto patrimônio, os valores logarítmicos por macro e a complexidade recebem mais importância.

Este é o principal local para modificar a modelagem se você quiser testar novas especificações.


In [23]:
def pesos_baixo_final():
    pesos = {}

    # Percentuais centrais
    for col in [
        "perc_ativos_financeiros",
        "perc_creditos_direitos",
        "perc_imoveis",
        "perc_participacoes_societarias",
        "perc_rural_agropecuario",
        "perc_veiculos"
    ]:
        pesos[col] = 0.75

    pesos["perc_dinheiro_especie"] = 0.45

    for col in [
        "perc_bens_luxo_colecao",
        "perc_direitos_intangiveis",
        "perc_outros",
        "perc_outros_atividade_profissional"
    ]:
        pesos[col] = 0.25

    # Valores log centrais
    for col in [
        "log_valor_ativos_financeiros",
        "log_valor_creditos_direitos",
        "log_valor_imoveis",
        "log_valor_participacoes_societarias",
        "log_valor_rural_agropecuario",
        "log_valor_veiculos"
    ]:
        pesos[col] = 1.00

    pesos["log_valor_dinheiro_especie"] = 0.60

    for col in [
        "log_valor_bens_luxo_colecao",
        "log_valor_direitos_intangiveis",
        "log_valor_outros",
        "log_valor_outros_atividade_profissional"
    ]:
        pesos[col] = 0.35

    # Complexidade
    pesos["log_qtd_bens"] = 0.80
    pesos["qtd_macros_presentes"] = 0.80
    pesos["indice_concentracao_macro"] = 0.80

    return pesos


def pesos_intermediario_final():
    pesos = {}

    for col in [
        "perc_ativos_financeiros",
        "perc_creditos_direitos",
        "perc_imoveis",
        "perc_participacoes_societarias",
        "perc_rural_agropecuario",
        "perc_veiculos"
    ]:
        pesos[col] = 0.75

    pesos["perc_dinheiro_especie"] = 0.35

    for col in [
        "perc_bens_luxo_colecao",
        "perc_direitos_intangiveis",
        "perc_outros",
        "perc_outros_atividade_profissional"
    ]:
        pesos[col] = 0.15

    for col in [
        "log_valor_ativos_financeiros",
        "log_valor_creditos_direitos",
        "log_valor_imoveis",
        "log_valor_participacoes_societarias",
        "log_valor_rural_agropecuario",
        "log_valor_veiculos"
    ]:
        pesos[col] = 1.00

    pesos["log_valor_dinheiro_especie"] = 0.40

    for col in [
        "log_valor_bens_luxo_colecao",
        "log_valor_direitos_intangiveis",
        "log_valor_outros",
        "log_valor_outros_atividade_profissional"
    ]:
        pesos[col] = 0.20

    pesos["log_qtd_bens"] = 0.85
    pesos["qtd_macros_presentes"] = 0.85
    pesos["indice_concentracao_macro"] = 0.85

    return pesos


def pesos_alto_final():
    pesos = {}

    for col in [
        "perc_ativos_financeiros",
        "perc_creditos_direitos",
        "perc_imoveis",
        "perc_participacoes_societarias",
        "perc_rural_agropecuario",
        "perc_veiculos"
    ]:
        pesos[col] = 0.70

    pesos["perc_dinheiro_especie"] = 0.30

    for col in [
        "perc_bens_luxo_colecao",
        "perc_direitos_intangiveis",
        "perc_outros",
        "perc_outros_atividade_profissional"
    ]:
        pesos[col] = 0.15

    for col in [
        "log_valor_ativos_financeiros",
        "log_valor_creditos_direitos",
        "log_valor_imoveis",
        "log_valor_participacoes_societarias",
        "log_valor_rural_agropecuario",
        "log_valor_veiculos"
    ]:
        pesos[col] = 1.10

    pesos["log_valor_dinheiro_especie"] = 0.35

    for col in [
        "log_valor_bens_luxo_colecao",
        "log_valor_direitos_intangiveis",
        "log_valor_outros",
        "log_valor_outros_atividade_profissional"
    ]:
        pesos[col] = 0.20

    pesos["log_qtd_bens"] = 1.00
    pesos["qtd_macros_presentes"] = 1.00
    pesos["indice_concentracao_macro"] = 1.00

    return pesos


In [24]:
# Tabela-documento dos pesos usados
pesos_documentacao = pd.DataFrame([
    {"estrato": "Baixo", "grupo_variavel": "Percentuais centrais", "peso": "0.75", "justificativa": "Composição patrimonial principal"},
    {"estrato": "Baixo", "grupo_variavel": "Percentual dinheiro", "peso": "0.45", "justificativa": "Categoria relevante, mas com influência moderada"},
    {"estrato": "Baixo", "grupo_variavel": "Percentuais raros/residuais", "peso": "0.25", "justificativa": "Reduzir isolamento por categorias raras"},
    {"estrato": "Baixo", "grupo_variavel": "Valores log centrais", "peso": "1.00", "justificativa": "Captar escala por macro"},
    {"estrato": "Baixo", "grupo_variavel": "Complexidade", "peso": "0.80", "justificativa": "Distinguir declarações simples e compostas"},
    {"estrato": "Intermediário", "grupo_variavel": "Percentuais centrais", "peso": "0.75", "justificativa": "Composição patrimonial principal"},
    {"estrato": "Intermediário", "grupo_variavel": "Percentuais raros/residuais", "peso": "0.15", "justificativa": "Evitar microclusters por exceções"},
    {"estrato": "Intermediário", "grupo_variavel": "Valores log centrais", "peso": "1.00", "justificativa": "Captar escala dentro da faixa"},
    {"estrato": "Intermediário", "grupo_variavel": "Complexidade", "peso": "0.85", "justificativa": "Distinguir perfis simples e diversificados"},
    {"estrato": "Alto/Ultra", "grupo_variavel": "Percentuais centrais", "peso": "0.70", "justificativa": "Composição principal, sem dominar a escala"},
    {"estrato": "Alto/Ultra", "grupo_variavel": "Valores log centrais", "peso": "1.10", "justificativa": "Maior importância da escala por macro"},
    {"estrato": "Alto/Ultra", "grupo_variavel": "Complexidade", "peso": "1.00", "justificativa": "Alta relevância da diversificação patrimonial"},
])

pesos_documentacao


,estrato,grupo_variavel,peso,justificativa
0,Baixo,Percentuais centrais,0.75,Composição patrimonial principal
1,Baixo,Percentual dinheiro,0.45,"Categoria relevante, mas com influência moderada"
2,Baixo,Percentuais raros/residuais,0.25,Reduzir isolamento por categorias raras
3,Baixo,Valores log centrais,1.00,Captar escala por macro
4,Baixo,Complexidade,0.80,Distinguir declarações simples e compostas
5,Intermediário,Percentuais centrais,0.75,Composição patrimonial principal
6,Intermediário,Percentuais raros/residuais,0.15,Evitar microclusters por exceções
7,Intermediário,Valores log centrais,1.00,Captar escala dentro da faixa
8,Intermediário,Complexidade,0.85,Distinguir perfis simples e diversificados
9,Alto/Ultra,Percentuais centrais,0.70,"Composição principal, sem dominar a escala"


## 6. Funções auxiliares de modelagem e métricas


In [25]:
def preparar_matriz_modelo(
    df,
    features,
    pesos,
    scaler="standard",
    clip=None
):
    # Padroniza, aplica pesos e opcionalmente limita extremos.
    X = df[features].copy()

    if scaler == "standard":
        scaler_obj = StandardScaler()
    elif scaler == "robust":
        scaler_obj = RobustScaler()
    else:
        raise ValueError("scaler deve ser 'standard' ou 'robust'.")

    X_scaled = pd.DataFrame(
        scaler_obj.fit_transform(X),
        columns=features,
        index=df.index
    )

    X_weighted = X_scaled.copy()

    for coluna, peso in pesos.items():
        if coluna not in X_weighted.columns:
            raise ValueError(f"Coluna de peso ausente da matriz: {coluna}")
        X_weighted[coluna] = X_weighted[coluna] * peso

    if clip is not None:
        X_weighted = X_weighted.clip(lower=clip[0], upper=clip[1])

    return X_weighted, scaler_obj


def rodar_kmeans(X, k, random_state=RANDOM_STATE, n_init=N_INIT_KMEANS):
    modelo = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=n_init
    )
    labels = modelo.fit_predict(X)
    return modelo, labels


def avaliar_kmeans(X, k_range, random_state=RANDOM_STATE, n_init=N_INIT_KMEANS):
    resultados = []

    for k in k_range:
        modelo, labels = rodar_kmeans(
            X,
            k=k,
            random_state=random_state,
            n_init=n_init
        )
        contagem = pd.Series(labels).value_counts()

        resultados.append({
            "k": k,
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "inertia": modelo.inertia_,
            "menor_cluster": contagem.min(),
            "maior_cluster": contagem.max(),
            "perc_maior_cluster": contagem.max() / len(labels)
        })

    return pd.DataFrame(resultados)


def resumir_clusters(df, coluna_cluster):
    resumo = (
        df
        .groupby(coluna_cluster)
        .agg(
            n_candidatos=("SQ_CANDIDATO", "count"),
            patrimonio_mediano=("patrimonio_total", "median"),
            patrimonio_medio=("patrimonio_total", "mean"),
            patrimonio_p25=("patrimonio_total", lambda x: x.quantile(0.25)),
            patrimonio_p75=("patrimonio_total", lambda x: x.quantile(0.75)),
            qtd_bens_media=("qtd_bens", "mean"),
            qtd_bens_mediana=("qtd_bens", "median"),
            qtd_macros_media=("qtd_macros_presentes", "mean"),
            concentracao_media=("indice_concentracao_macro", "mean"),
            perc_imoveis=("perc_imoveis", "mean"),
            perc_veiculos=("perc_veiculos", "mean"),
            perc_ativos_financeiros=("perc_ativos_financeiros", "mean"),
            perc_participacoes=("perc_participacoes_societarias", "mean"),
            perc_rural=("perc_rural_agropecuario", "mean"),
            perc_creditos=("perc_creditos_direitos", "mean"),
            perc_dinheiro=("perc_dinheiro_especie", "mean"),
            perc_outros=("perc_outros", "mean")
        )
        .round(3)
    )

    perc_cols_resumo = [
        "perc_imoveis",
        "perc_veiculos",
        "perc_ativos_financeiros",
        "perc_participacoes",
        "perc_rural",
        "perc_creditos",
        "perc_dinheiro",
        "perc_outros"
    ]

    dominancia = pd.DataFrame({
        "macro_dominante": resumo[perc_cols_resumo].idxmax(axis=1),
        "percentual_medio": resumo[perc_cols_resumo].max(axis=1)
    })

    return resumo, dominancia


## 7. Preparar matrizes finais por estrato


In [26]:
# ============================================================
# Baixo patrimônio
# ============================================================

df_baixo = features_candidatos[
    features_candidatos["estrato_base"].eq("Baixo patrimônio")
].copy()

X_baixo_final, scaler_baixo_final = preparar_matriz_modelo(
    df=df_baixo,
    features=features_modelo,
    pesos=pesos_baixo_final(),
    scaler="standard",
    clip=None
)

print("Baixo patrimônio:", df_baixo.shape)


Baixo patrimônio: (4806, 43)


In [27]:
# ============================================================
# Patrimônio intermediário
# ============================================================

df_intermediario = features_candidatos[
    features_candidatos["estrato_base"].eq("Patrimônio intermediário")
].copy()

X_intermediario_final, scaler_intermediario_final = preparar_matriz_modelo(
    df=df_intermediario,
    features=features_modelo,
    pesos=pesos_intermediario_final(),
    scaler="standard",
    clip=None
)

print("Patrimônio intermediário:", df_intermediario.shape)


Patrimônio intermediário: (7970, 43)


In [28]:
# ============================================================
# Alto patrimônio regular e ultra-alto patrimônio
# ============================================================

df_alto = features_candidatos[
    features_candidatos["estrato_base"].eq("Alto patrimônio")
].copy()

# Garantir subestrato no dataframe separado
df_alto["subestrato_alto"] = pd.cut(
    df_alto["patrimonio_total"],
    bins=[CORTE_INTERMEDIARIO, CORTE_ULTRA_ALTO, np.inf],
    labels=[
        "Alto patrimônio",
        "Ultra-alto patrimônio"
    ],
    include_lowest=True
).astype("string")

df_alto_regular = df_alto[
    df_alto["subestrato_alto"].eq("Alto patrimônio")
].copy()

df_ultra = df_alto[
    df_alto["subestrato_alto"].eq("Ultra-alto patrimônio")
].copy()

X_alto_regular_final, scaler_alto_regular_final = preparar_matriz_modelo(
    df=df_alto_regular,
    features=features_modelo,
    pesos=pesos_alto_final(),
    scaler="robust",
    clip=CLIP_ALTO
)

X_ultra_final, scaler_ultra_final = preparar_matriz_modelo(
    df=df_ultra,
    features=features_modelo,
    pesos=pesos_alto_final(),
    scaler="robust",
    clip=CLIP_ALTO
)

print("Alto patrimônio regular:", df_alto_regular.shape)
print("Ultra-alto patrimônio:", df_ultra.shape)


Alto patrimônio regular: (4484, 43)
Ultra-alto patrimônio: (959, 43)


## 8. Métricas de clustering — opcional

A célula abaixo permite recalcular métricas para diferentes valores de `k`. Por padrão, `RODAR_BUSCAS_METRICAS = False` para evitar tempo de execução desnecessário.

Para testar novas soluções, altere `RODAR_BUSCAS_METRICAS = True` na seção de configuração ou rode cada chamada manualmente.


In [29]:
if RODAR_BUSCAS_METRICAS:
    avaliacao_baixo = avaliar_kmeans(X_baixo_final, RANGES_K["baixo"])
    avaliacao_intermediario = avaliar_kmeans(X_intermediario_final, RANGES_K["intermediario"])
    avaliacao_alto_regular = avaliar_kmeans(X_alto_regular_final, RANGES_K["alto_regular"])
    avaliacao_ultra = avaliar_kmeans(X_ultra_final, RANGES_K["ultra"])

    display(avaliacao_baixo)
    display(avaliacao_intermediario)
    display(avaliacao_alto_regular)
    display(avaliacao_ultra)
else:
    print("Busca de métricas não executada. Altere RODAR_BUSCAS_METRICAS para True se quiser recalcular.")


Busca de métricas não executada. Altere RODAR_BUSCAS_METRICAS para True se quiser recalcular.


## 9. Rodar a solução final


In [31]:
# ============================================================
# K-Means final por estrato
# ============================================================

modelo_baixo_final, labels_baixo = rodar_kmeans(X_baixo_final, K_BAIXO)
df_baixo["cluster_baixo"] = labels_baixo

modelo_intermediario_final, labels_intermediario = rodar_kmeans(X_intermediario_final, K_INTERMEDIARIO)
df_intermediario["cluster_intermediario"] = labels_intermediario

modelo_alto_regular_final, labels_alto_regular = rodar_kmeans(X_alto_regular_final, K_ALTO_REGULAR)
df_alto_regular["cluster_alto"] = labels_alto_regular

modelo_ultra_final, labels_ultra = rodar_kmeans(X_ultra_final, K_ULTRA)
df_ultra["cluster_ultra_alto"] = labels_ultra

print("Distribuição — baixo patrimônio")
print(df_baixo["cluster_baixo"].value_counts().sort_index())

print("\nDistribuição — patrimônio intermediário")
print(df_intermediario["cluster_intermediario"].value_counts().sort_index())

print("\nDistribuição — alto patrimônio")
print(df_alto_regular["cluster_alto"].value_counts().sort_index())

print("\nDistribuição — ultra-alto patrimônio")
print(df_ultra["cluster_ultra_alto"].value_counts().sort_index())


Distribuição — baixo patrimônio
cluster_baixo
0    2017
1     693
2     343
3     552
4      88
5      52
6     652
7     409
Name: count, dtype: int64

Distribuição — patrimônio intermediário
cluster_intermediario
0    2239
1     521
2     499
3     282
4     361
5    1750
6     944
7     652
8     498
9     224
Name: count, dtype: int64

Distribuição — alto patrimônio
cluster_alto
0     173
1    1832
2     482
3     611
4     192
5     347
6     384
7     271
8     192
Name: count, dtype: int64

Distribuição — ultra-alto patrimônio
cluster_ultra_alto
0    172
1    104
2    330
3    112
4     72
5     76
6     93
Name: count, dtype: int64


## 10. Nomear perfis finais


In [32]:
# ============================================================
# Nomes finais dos perfis - geração automática hierárquica
# ============================================================

# A regra abaixo evita depender do número arbitrário do cluster do K-Means.
# Todo cluster é nomeado por:
#   1) estrato patrimonial;
#   2) macro patrimonial mais dominante;
#   3) grau de concentração, macro complementar ou perfil misto.
#
# Regra hierárquica:
# - se macro_1 >= 70%:
#       "estrato - macro_1 concentrado"
# - senão, se macro_1 + macro_2 >= 70% E macro_2 >= 15%:
#       "estrato - macro_1 macro_2"
# - senão:
#       "estrato - macro_1 misto"
#
# Regra adicional:
# - se dois clusters do mesmo estrato gerarem o mesmo nome-base,
#   o nome é desambiguado com macro_2, macro_3 e, se necessário,
#   com grau relativo de concentração.
#
# Objetivo:
# - evitar nomes como:
#       "Alto patrimônio - imobiliário misto 1"
#       "Alto patrimônio - imobiliário misto 2"
# - preferir nomes explicativos como:
#       "Alto patrimônio - imobiliário rural agropecuário misto"
#       "Alto patrimônio - imobiliário financeiro misto"
#
# Observação:
# - Os hífens foram removidos dos nomes das macros.
# - O separador entre estrato e nome-base continua " - " para legibilidade.

LIMIAR_CONCENTRADO = 0.70
LIMIAR_COMBINADO = 0.70
LIMIAR_MACRO_COMPLEMENTAR = 0.15

macro_perc_cols = [
    "perc_imoveis",
    "perc_veiculos",
    "perc_ativos_financeiros",
    "perc_participacoes_societarias",
    "perc_rural_agropecuario",
    "perc_creditos_direitos",
    "perc_dinheiro_especie",
    "perc_outros",
    "perc_bens_luxo_colecao",
    "perc_direitos_intangiveis",
    "perc_outros_atividade_profissional",
]

nomes_macro = {
    "perc_imoveis": "imobiliário",
    "perc_veiculos": "veicular",
    "perc_ativos_financeiros": "financeiro",
    "perc_participacoes_societarias": "societário",
    "perc_rural_agropecuario": "rural agropecuário",
    "perc_creditos_direitos": "créditos direitos",
    "perc_dinheiro_especie": "dinheiro espécie",
    "perc_outros": "outros ativos",
    "perc_bens_luxo_colecao": "bens luxo coleção",
    "perc_direitos_intangiveis": "direitos intangíveis",
    "perc_outros_atividade_profissional": "atividade profissional",
}


def gerar_nomes_clusters_automaticos(
    df_estrato,
    coluna_cluster,
    nome_estrato,
    prefixo_codigo,
    limiar_concentrado=LIMIAR_CONCENTRADO,
    limiar_combinado=LIMIAR_COMBINADO,
    limiar_macro_complementar=LIMIAR_MACRO_COMPLEMENTAR,
):
    """Gera nomes automáticos seguindo a regra macro dominante -> concentrado/complementar/misto."""

    resumo = (
        df_estrato
        .groupby(coluna_cluster)
        .agg(
            n_candidatos=("SQ_CANDIDATO", "count"),
            patrimonio_mediano=("patrimonio_total", "median"),
            patrimonio_medio=("patrimonio_total", "mean"),
            qtd_macros_media=("qtd_macros_presentes", "mean"),
            concentracao_media=("indice_concentracao_macro", "mean"),
            **{col: (col, "mean") for col in macro_perc_cols}
        )
        .reset_index()
    )

    macro_cols_existentes = [col for col in macro_perc_cols if col in resumo.columns]

    macro_1_col = []
    macro_1_pct = []
    macro_2_col = []
    macro_2_pct = []
    macro_3_col = []
    macro_3_pct = []

    for _, row in resumo[macro_cols_existentes].iterrows():
        ordenado = row.sort_values(ascending=False)

        macro_1_col.append(ordenado.index[0])
        macro_1_pct.append(ordenado.iloc[0])

        if len(ordenado) > 1:
            macro_2_col.append(ordenado.index[1])
            macro_2_pct.append(ordenado.iloc[1])
        else:
            macro_2_col.append(pd.NA)
            macro_2_pct.append(np.nan)

        if len(ordenado) > 2:
            macro_3_col.append(ordenado.index[2])
            macro_3_pct.append(ordenado.iloc[2])
        else:
            macro_3_col.append(pd.NA)
            macro_3_pct.append(np.nan)

    resumo["macro_1_col"] = macro_1_col
    resumo["macro_1_pct"] = macro_1_pct
    resumo["macro_2_col"] = macro_2_col
    resumo["macro_2_pct"] = macro_2_pct
    resumo["macro_3_col"] = macro_3_col
    resumo["macro_3_pct"] = macro_3_pct

    resumo["macro_1_2_pct"] = resumo["macro_1_pct"] + resumo["macro_2_pct"]

    resumo["macro_1"] = resumo["macro_1_col"].map(nomes_macro)
    resumo["macro_2"] = resumo["macro_2_col"].map(nomes_macro)
    resumo["macro_3"] = resumo["macro_3_col"].map(nomes_macro)

    cond_concentrado = resumo["macro_1_pct"].ge(limiar_concentrado)

    cond_complementar = (
        ~cond_concentrado
        & resumo["macro_1_2_pct"].ge(limiar_combinado)
        & resumo["macro_2_pct"].ge(limiar_macro_complementar)
    )

    resumo["tipo_nome"] = np.select(
        [cond_concentrado, cond_complementar],
        ["concentrado", "macro_complementar"],
        default="misto"
    )

    resumo["nome_base"] = np.select(
        [cond_concentrado, cond_complementar],
        [
            resumo["macro_1"] + " concentrado",
            resumo["macro_1"] + " " + resumo["macro_2"],
        ],
        default=resumo["macro_1"] + " misto"
    )

    resumo["nome_base_original"] = resumo["nome_base"]

    # Identifica nomes repetidos dentro do estrato.
    resumo["nome_base_repetido"] = (
        resumo.groupby("nome_base_original")[coluna_cluster].transform("count") > 1
    )

    def montar_nome_desambiguado(row):
        macro_1 = row["macro_1"]
        macro_2 = row["macro_2"]
        macro_3 = row["macro_3"]
        tipo = row["tipo_nome"]

        if tipo == "concentrado":
            # Se dois clusters são concentrados na mesma macro,
            # usamos a macro secundária para explicar a composição residual.
            if pd.notna(macro_2):
                return f"{macro_1} concentrado com {macro_2}"
            return f"{macro_1} concentrado"

        if tipo == "macro_complementar":
            # A regra principal já usa macro_1 + macro_2.
            # Se houver duplicidade, adicionamos macro_3 como nuance.
            if pd.notna(macro_3):
                return f"{macro_1} {macro_2} com {macro_3}"
            return f"{macro_1} {macro_2}"

        # Para perfis mistos, adicionamos macro_2 e macro_3 ao nome,
        # pois "macro_1 misto" costuma ser genérico demais.
        if pd.notna(macro_2) and pd.notna(macro_3):
            return f"{macro_1} {macro_2} {macro_3} misto"

        if pd.notna(macro_2):
            return f"{macro_1} {macro_2} misto"

        return f"{macro_1} misto"

    # Aplica desambiguação apenas nos nomes repetidos.
    resumo.loc[resumo["nome_base_repetido"], "nome_base"] = resumo.loc[
        resumo["nome_base_repetido"]
    ].apply(montar_nome_desambiguado, axis=1)

    # Verifica se ainda sobrou duplicidade após macro_2 e macro_3.
    resumo["nome_base_ainda_repetido"] = (
        resumo.groupby("nome_base")[coluna_cluster].transform("count") > 1
    )

    # Se ainda houver repetição, diferencia pela concentração relativa,
    # sem recorrer a números.
    if resumo["nome_base_ainda_repetido"].any():
        resumo = resumo.sort_values(
            ["nome_base", "concentracao_media", "patrimonio_mediano", "n_candidatos", coluna_cluster],
            ascending=[True, False, True, False, True]
        ).copy()

        resumo["ranking_concentracao"] = (
            resumo.groupby("nome_base").cumcount()
        )

        resumo.loc[
            resumo["nome_base_ainda_repetido"] & resumo["ranking_concentracao"].eq(0),
            "nome_base"
        ] = (
            resumo.loc[
                resumo["nome_base_ainda_repetido"] & resumo["ranking_concentracao"].eq(0),
                "nome_base"
            ]
            + " mais concentrado"
        )

        resumo.loc[
            resumo["nome_base_ainda_repetido"] & resumo["ranking_concentracao"].gt(0),
            "nome_base"
        ] = (
            resumo.loc[
                resumo["nome_base_ainda_repetido"] & resumo["ranking_concentracao"].gt(0),
                "nome_base"
            ]
            + " mais diversificado"
        )
    else:
        resumo["ranking_concentracao"] = np.nan

    # Checagem final de duplicidade no estrato.
    resumo["duplicado_no_estrato"] = (
        resumo.groupby("nome_base")[coluna_cluster].transform("count") > 1
    )

    resumo["perfil_automatico"] = nome_estrato + " - " + resumo["nome_base"]

    resumo["estrato"] = nome_estrato
    resumo["cluster_final_codigo"] = prefixo_codigo + resumo[coluna_cluster].astype(str)

    mapa_nomes = dict(zip(resumo[coluna_cluster], resumo["perfil_automatico"]))

    colunas_auditoria = [
        "estrato",
        "cluster_final_codigo",
        coluna_cluster,
        "perfil_automatico",
        "nome_base",
        "nome_base_original",
        "tipo_nome",
        "macro_1",
        "macro_1_pct",
        "macro_2",
        "macro_2_pct",
        "macro_3",
        "macro_3_pct",
        "macro_1_2_pct",
        "nome_base_repetido",
        "nome_base_ainda_repetido",
        "duplicado_no_estrato",
        "ranking_concentracao",
        "n_candidatos",
        "patrimonio_mediano",
        "patrimonio_medio",
        "qtd_macros_media",
        "concentracao_media",
    ]

    return mapa_nomes, resumo[colunas_auditoria].sort_values(
        ["estrato", "perfil_automatico"]
    ).reset_index(drop=True)


mapa_baixo, auditoria_baixo = gerar_nomes_clusters_automaticos(
    df_baixo,
    coluna_cluster="cluster_baixo",
    nome_estrato="Baixo patrimônio",
    prefixo_codigo="baixo_k8_c",
)

mapa_intermediario, auditoria_intermediario = gerar_nomes_clusters_automaticos(
    df_intermediario,
    coluna_cluster="cluster_intermediario",
    nome_estrato="Patrimônio intermediário",
    prefixo_codigo="intermediario_k10_c",
)

mapa_alto_regular, auditoria_alto_regular = gerar_nomes_clusters_automaticos(
    df_alto_regular,
    coluna_cluster="cluster_alto",
    nome_estrato="Alto patrimônio",
    prefixo_codigo="alto_k9_c",
)

mapa_ultra, auditoria_ultra = gerar_nomes_clusters_automaticos(
    df_ultra,
    coluna_cluster="cluster_ultra_alto",
    nome_estrato="Ultra alto patrimônio",
    prefixo_codigo="ultra_k7_c",
)

# Aplicar nomes automáticos
df_baixo["perfil_baixo"] = df_baixo["cluster_baixo"].map(mapa_baixo)
df_intermediario["perfil_intermediario"] = df_intermediario["cluster_intermediario"].map(mapa_intermediario)
df_alto_regular["perfil_alto_regular"] = df_alto_regular["cluster_alto"].map(mapa_alto_regular)
df_ultra["perfil_ultra"] = df_ultra["cluster_ultra_alto"].map(mapa_ultra)

# Tabela de auditoria dos nomes gerados automaticamente.
nomes_clusters_automaticos = pd.concat(
    [auditoria_baixo, auditoria_intermediario, auditoria_alto_regular, auditoria_ultra],
    ignore_index=True
)

nomes_clusters_automaticos

,estrato,cluster_final_codigo,cluster_baixo,perfil_automatico,nome_base,nome_base_original,tipo_nome,macro_1,macro_1_pct,macro_2,...,duplicado_no_estrato,ranking_concentracao,n_candidatos,patrimonio_mediano,patrimonio_medio,qtd_macros_media,concentracao_media,cluster_intermediario,cluster_alto,cluster_ultra_alto
0,Baixo patrimônio,baixo_k8_c5,5.0,Baixo patrimônio - créditos direitos financeiro,créditos direitos financeiro,créditos direitos financeiro,macro_complementar,créditos direitos,0.543511,financeiro,...,False,NaN,52,509355.5,4.653524e+05,2.115385,0.713278,NaN,NaN,NaN
1,Baixo patrimônio,baixo_k8_c7,7.0,Baixo patrimônio - dinheiro espécie outros ativos,dinheiro espécie outros ativos,dinheiro espécie outros ativos,macro_complementar,dinheiro espécie,0.660300,outros ativos,...,False,NaN,409,90000.0,1.716313e+05,1.051345,0.994510,NaN,NaN,NaN
2,Baixo patrimônio,baixo_k8_c3,3.0,Baixo patrimônio - financeiro concentrado,financeiro concentrado,financeiro concentrado,concentrado,financeiro,0.990217,societário,...,False,NaN,552,99427.5,2.118396e+05,1.083333,0.988962,NaN,NaN,NaN
3,Baixo patrimônio,baixo_k8_c1,1.0,Baixo patrimônio - imobiliário concentrado,imobiliário concentrado,imobiliário concentrado,concentrado,imobiliário,0.969706,veicular,...,False,NaN,693,630000.0,6.171408e+05,1.235209,0.951228,NaN,NaN,NaN
4,Baixo patrimônio,baixo_k8_c4,4.0,Baixo patrimônio - rural agropecuário veicular,rural agropecuário veicular,rural agropecuário veicular,macro_complementar,rural agropecuário,0.699086,veicular,...,False,NaN,88,629880.0,6.181852e+05,1.818182,0.749315,NaN,NaN,NaN
5,Baixo patrimônio,baixo_k8_c2,2.0,Baixo patrimônio - societário concentrado,societário concentrado,societário concentrado,concentrado,societário,0.952410,veicular,...,False,NaN,343,200000.0,3.577097e+05,1.192420,0.947301,NaN,NaN,NaN
6,Baixo patrimônio,baixo_k8_c0,0.0,Baixo patrimônio - veicular concentrado,veicular concentrado,veicular concentrado,concentrado,veicular,0.999499,dinheiro espécie,...,False,NaN,2017,292270.0,3.447319e+05,1.015369,0.999062,NaN,NaN,NaN
7,Baixo patrimônio,baixo_k8_c6,6.0,Baixo patrimônio - veicular financeiro,veicular financeiro,veicular financeiro,macro_complementar,veicular,0.535827,financeiro,...,False,NaN,652,550000.0,5.440544e+05,2.239264,0.635021,NaN,NaN,NaN
8,Patrimônio intermediário,intermediario_k10_c3,NaN,Patrimônio intermediário - créditos direitos m...,créditos direitos misto,créditos direitos misto,misto,créditos direitos,0.344765,imobiliário,...,False,NaN,282,4542670.5,4.900636e+06,3.421986,0.553693,3.0,NaN,NaN
9,Patrimônio intermediário,intermediario_k10_c8,NaN,Patrimônio intermediário - financeiro concentrado,financeiro concentrado,financeiro concentrado,concentrado,financeiro,0.813650,veicular,...,False,NaN,498,2601391.0,3.515849e+06,1.851406,0.787584,8.0,NaN,NaN


## 11. Resumos técnicos dos clusters finais

Estas tabelas são úteis para conferir se a solução continua reproduzindo os perfis esperados.


In [35]:
resumo_baixo, dominancia_baixo = resumir_clusters(df_baixo, "cluster_baixo")
resumo_intermediario, dominancia_intermediario = resumir_clusters(df_intermediario, "cluster_intermediario")
resumo_alto_regular, dominancia_alto_regular = resumir_clusters(df_alto_regular, "cluster_alto")
resumo_ultra, dominancia_ultra = resumir_clusters(df_ultra, "cluster_ultra_alto")

print("Resumo baixo patrimônio")
display(resumo_baixo)
print("Dominância baixo patrimônio")
display(dominancia_baixo)

print("Resumo patrimônio intermediário")
display(resumo_intermediario)
print("Dominância patrimônio intermediário")
display(dominancia_intermediario)

print("Resumo alto patrimônio")
display(resumo_alto_regular)
print("Dominância alto patrimônio")
display(dominancia_alto_regular)

print("Resumo ultra-alto patrimônio")
display(resumo_ultra)
print("Dominância ultra-alto patrimônio")
display(dominancia_ultra)


Resumo baixo patrimônio


,n_candidatos,patrimonio_mediano,patrimonio_medio,patrimonio_p25,patrimonio_p75,qtd_bens_media,qtd_bens_mediana,qtd_macros_media,concentracao_media,perc_imoveis,perc_veiculos,perc_ativos_financeiros,perc_participacoes,perc_rural,perc_creditos,perc_dinheiro,perc_outros
cluster_baixo,,,,,,,,,,,,,,,,,
0,2017,292270.0,344731.931,150000.0,500000.00,1.235,1.0,1.015,0.999,0.000,0.999,0.000,0.000,0.000,0.000,0.000,0.000
1,693,630000.0,617140.779,400000.0,830000.00,1.371,1.0,1.235,0.951,0.970,0.019,0.002,0.004,0.000,0.000,0.003,0.001
2,343,200000.0,357709.738,99000.0,600000.00,1.402,1.0,1.192,0.947,0.007,0.017,0.007,0.952,0.000,0.000,0.015,0.001
3,552,99427.5,211839.603,20000.0,329662.50,1.775,1.0,1.083,0.989,0.000,0.001,0.990,0.003,0.000,0.000,0.001,0.003
4,88,629880.0,618185.227,414900.0,835000.00,2.205,2.0,1.818,0.749,0.053,0.188,0.013,0.021,0.699,0.000,0.013,0.012
5,52,509355.5,465352.423,143353.0,728088.00,2.750,2.0,2.115,0.713,0.026,0.141,0.158,0.059,0.008,0.544,0.023,0.041
6,652,550000.0,544054.417,320448.5,760916.75,2.994,3.0,2.239,0.635,0.142,0.536,0.178,0.043,0.000,0.000,0.071,0.023
7,409,90000.0,171631.311,30000.0,200000.00,1.122,1.0,1.051,0.995,0.001,0.000,0.001,0.001,0.000,0.000,0.660,0.306


Dominância baixo patrimônio


,macro_dominante,percentual_medio
cluster_baixo,,
0,perc_veiculos,0.999
1,perc_imoveis,0.970
2,perc_participacoes,0.952
3,perc_ativos_financeiros,0.990
4,perc_rural,0.699
5,perc_creditos,0.544
6,perc_veiculos,0.536
7,perc_dinheiro,0.660


Resumo patrimônio intermediário


...

,macro_dominante,percentual_medio
cluster_alto,,
0,perc_veiculos,0.605
1,perc_imoveis,0.760
2,perc_imoveis,0.376
3,perc_imoveis,0.487
4,perc_imoveis,0.383
5,perc_ativos_financeiros,0.510
6,perc_imoveis,0.585
7,perc_imoveis,0.441
8,perc_imoveis,0.303


Resumo ultra-alto patrimônio


,n_candidatos,patrimonio_mediano,patrimonio_medio,patrimonio_p25,patrimonio_p75,qtd_bens_media,qtd_bens_mediana,qtd_macros_media,concentracao_media,perc_imoveis,perc_veiculos,perc_ativos_financeiros,perc_participacoes,perc_rural,perc_creditos,perc_dinheiro,perc_outros
cluster_ultra_alto,,,,,,,,,,,,,,,,,
0,172,182120332.0,4.249514e+08,1.330983e+08,3.046185e+08,26.733,23.0,5.831,0.595,0.426,0.028,0.351,0.071,0.076,0.011,0.024,0.012
1,104,176628293.5,3.563462e+08,1.324167e+08,2.815094e+08,20.663,17.0,4.327,0.584,0.201,0.024,0.121,0.046,0.597,0.004,0.001,0.007
2,330,174298679.0,3.060533e+08,1.256426e+08,2.909432e+08,17.615,14.0,4.079,0.698,0.483,0.018,0.442,0.043,0.008,0.005,0.000,0.000
3,112,261137176.5,9.959062e+08,1.409233e+08,7.062614e+08,25.393,20.0,5.741,0.481,0.196,0.022,0.237,0.115,0.045,0.378,0.003,0.001
4,72,161368476.0,1.952439e+08,1.311029e+08,2.158013e+08,8.236,7.0,2.708,0.899,0.931,0.020,0.000,0.016,0.004,0.006,0.001,0.012
5,76,291240263.5,3.371487e+09,1.467742e+08,7.519738e+08,17.789,11.5,4.026,0.724,0.079,0.010,0.063,0.812,0.014,0.018,0.003,0.001
6,93,206173452.0,4.665089e+08,1.402661e+08,4.818180e+08,26.290,21.0,5.183,0.541,0.308,0.025,0.243,0.120,0.062,0.049,0.000,0.191


Dominância ultra-alto patrimônio


,macro_dominante,percentual_medio
cluster_ultra_alto,,
0,perc_imoveis,0.426
1,perc_rural,0.597
2,perc_imoveis,0.483
3,perc_creditos,0.378
4,perc_imoveis,0.931
5,perc_participacoes,0.812
6,perc_imoveis,0.308


## 12. Consolidar resultado final no dataframe principal

Esta seção gera as colunas finais:

- `estrato_final`
- `cluster_final_codigo`
- `perfil_patrimonial_final`

Essas colunas devem ser usadas nos notebooks de análise complementar.


In [38]:
# ============================================================
# Consolidar perfis finais estratificados
# ============================================================

features_candidatos["estrato_final"] = pd.Series(
    pd.NA,
    index=features_candidatos.index,
    dtype="string"
)

features_candidatos["cluster_final_codigo"] = pd.Series(
    pd.NA,
    index=features_candidatos.index,
    dtype="string"
)

features_candidatos["perfil_patrimonial_final"] = pd.Series(
    pd.NA,
    index=features_candidatos.index,
    dtype="string"
)

# Baixo patrimônio
features_candidatos.loc[df_baixo.index, "estrato_final"] = "Baixo patrimônio"
features_candidatos.loc[df_baixo.index, "cluster_final_codigo"] = (
    "baixo_k8_c" + df_baixo["cluster_baixo"].astype(str)
)
features_candidatos.loc[df_baixo.index, "perfil_patrimonial_final"] = (
    df_baixo["perfil_baixo"].astype("string")
)

# Patrimônio intermediário
features_candidatos.loc[df_intermediario.index, "estrato_final"] = "Patrimônio intermediário"
features_candidatos.loc[df_intermediario.index, "cluster_final_codigo"] = (
    "intermediario_k10_c" + df_intermediario["cluster_intermediario"].astype(str)
)
features_candidatos.loc[df_intermediario.index, "perfil_patrimonial_final"] = (
    df_intermediario["perfil_intermediario"].astype("string")
)

# Alto patrimônio regular
features_candidatos.loc[df_alto_regular.index, "estrato_final"] = "Alto patrimônio"
features_candidatos.loc[df_alto_regular.index, "cluster_final_codigo"] = (
    "alto_k9_c" + df_alto_regular["cluster_alto"].astype(str)
)
features_candidatos.loc[df_alto_regular.index, "perfil_patrimonial_final"] = (
    df_alto_regular["perfil_alto_regular"].astype("string")
)

# Ultra-alto patrimônio
features_candidatos.loc[df_ultra.index, "estrato_final"] = "Ultra-alto patrimônio"
features_candidatos.loc[df_ultra.index, "cluster_final_codigo"] = (
    "ultra_k7_c" + df_ultra["cluster_ultra_alto"].astype(str)
)
features_candidatos.loc[df_ultra.index, "perfil_patrimonial_final"] = (
    df_ultra["perfil_ultra"].astype("string")
)

# Conferência
features_candidatos[[
    "estrato_final",
    "cluster_final_codigo",
    "perfil_patrimonial_final"
]].isna().sum()


estrato_final               0
cluster_final_codigo        0
perfil_patrimonial_final    0
dtype: int64

In [39]:
print("Distribuição dos estratos finais")
display(features_candidatos["estrato_final"].value_counts(dropna=False))

print("Distribuição dos perfis finais")
display(features_candidatos["perfil_patrimonial_final"].value_counts(dropna=False))


Distribuição dos estratos finais


estrato_final
Patrimônio intermediário    7970
Baixo patrimônio            4806
Alto patrimônio             4484
Ultra-alto patrimônio        959
Name: count, dtype: Int64

Distribuição dos perfis finais


perfil_patrimonial_final
Patrimônio intermediário - imobiliário concentrado com veicular                         2239
Baixo patrimônio - veicular concentrado                                                 2017
Alto patrimônio - imobiliário concentrado                                               1832
Patrimônio intermediário - imobiliário concentrado com outros ativos                    1750
Patrimônio intermediário - imobiliário financeiro                                        944
Baixo patrimônio - imobiliário concentrado                                               693
Baixo patrimônio - veicular financeiro                                                   652
Patrimônio intermediário - imobiliário misto                                             652
Alto patrimônio - imobiliário rural agropecuário                                         611
Baixo patrimônio - financeiro concentrado                                                552
Patrimônio intermediário - veicular concentra

## 13. Tabelas finais de resumo


In [40]:
resumo_perfis_finais = (
    features_candidatos
    .groupby(["estrato_final", "cluster_final_codigo", "perfil_patrimonial_final"])
    .agg(
        n_candidatos=("SQ_CANDIDATO", "count"),
        patrimonio_mediano=("patrimonio_total", "median"),
        patrimonio_medio=("patrimonio_total", "mean"),
        patrimonio_p25=("patrimonio_total", lambda x: x.quantile(0.25)),
        patrimonio_p75=("patrimonio_total", lambda x: x.quantile(0.75)),
        qtd_bens_mediana=("qtd_bens", "median"),
        qtd_bens_media=("qtd_bens", "mean"),
        qtd_macros_media=("qtd_macros_presentes", "mean"),
        concentracao_media=("indice_concentracao_macro", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values(["estrato_final", "n_candidatos"], ascending=[True, False])
)

resumo_perfis_finais


,estrato_final,cluster_final_codigo,perfil_patrimonial_final,n_candidatos,patrimonio_mediano,patrimonio_medio,patrimonio_p25,patrimonio_p75,qtd_bens_mediana,qtd_bens_media,qtd_macros_media,concentracao_media
1,Alto patrimônio,alto_k9_c1,Alto patrimônio - imobiliário concentrado,1832,22205598.5,3.052906e+07,1.414679e+07,4.011324e+07,5.0,6.37,2.59,0.78
3,Alto patrimônio,alto_k9_c3,Alto patrimônio - imobiliário rural agropecuário,611,26000000.0,3.464253e+07,1.577500e+07,4.733367e+07,8.0,9.92,3.85,0.55
2,Alto patrimônio,alto_k9_c2,Alto patrimônio - imobiliário financeiro crédi...,482,32066501.0,3.755578e+07,1.762704e+07,5.196042e+07,9.0,10.73,4.30,0.57
6,Alto patrimônio,alto_k9_c6,Alto patrimônio - imobiliário financeiro,384,25663259.5,3.484807e+07,1.547411e+07,4.889611e+07,8.0,9.57,4.12,0.62
5,Alto patrimônio,alto_k9_c5,Alto patrimônio - financeiro societário,347,21391675.0,2.855804e+07,1.451406e+07,3.531564e+07,4.0,6.56,2.33,0.82
7,Alto patrimônio,alto_k9_c7,Alto patrimônio - imobiliário societário,271,24780117.0,3.203187e+07,1.531714e+07,4.111816e+07,8.0,8.94,3.48,0.51
4,Alto patrimônio,alto_k9_c4,Alto patrimônio - imobiliário rural agropecuár...,192,28244319.5,3.591225e+07,1.659681e+07,5.095584e+07,11.0,12.10,5.07,0.49
8,Alto patrimônio,alto_k9_c8,Alto patrimônio - imobiliário rural agropecuár...,192,36064147.5,4.244196e+07,2.104762e+07,6.164135e+07,14.0,15.92,5.59,0.45
0,Alto patrimônio,alto_k9_c0,Alto patrimônio - veicular imobiliário,173,16047335.0,2.015283e+07,1.197376e+07,2.275151e+07,5.0,6.56,2.94,0.58
9,Baixo patrimônio,baixo_k8_c0,Baixo patrimônio - veicular concentrado,2017,292270.0,3.447319e+05,1.500000e+05,5.000000e+05,1.0,1.23,1.02,1.00


In [41]:
clusters_por_candidato = (
    features_candidatos[
        [
            "SQ_CANDIDATO",
            "SG_UF",
            "estrato_final",
            "cluster_final_codigo",
            "perfil_patrimonial_final"
        ]
    ]
    .drop_duplicates(subset=["SQ_CANDIDATO"])
    .copy()
)

clusters_por_candidato.head()


,SQ_CANDIDATO,SG_UF,estrato_final,cluster_final_codigo,perfil_patrimonial_final
0,10001595335,AC,Baixo patrimônio,baixo_k8_c0,Baixo patrimônio - veicular concentrado
1,10001595336,AC,Patrimônio intermediário,intermediario_k10_c0,Patrimônio intermediário - imobiliário concent...
2,10001595338,AC,Patrimônio intermediário,intermediario_k10_c0,Patrimônio intermediário - imobiliário concent...
3,10001595339,AC,Patrimônio intermediário,intermediario_k10_c0,Patrimônio intermediário - imobiliário concent...
4,10001595340,AC,Baixo patrimônio,baixo_k8_c0,Baixo patrimônio - veicular concentrado


## 14. Exportar resultados finais

Os arquivos exportados serão usados nos próximos notebooks de análise complementar.


In [42]:
# Dataset completo de features com clusters finais
features_candidatos.to_csv(
    "features_candidatos_com_clusters_patrimoniais.csv",
    index=False,
    sep=";"
)

# Versão parquet, se o ambiente tiver suporte
try:
    features_candidatos.to_parquet(
        "features_candidatos_com_clusters_patrimoniais.parquet",
        index=False
    )
except Exception as e:
    print("Parquet não exportado:", e)

# Tabela auxiliar por candidato
clusters_por_candidato.to_csv(
    "clusters_patrimoniais_por_candidato.csv",
    index=False,
    sep=";"
)

try:
    clusters_por_candidato.to_parquet(
        "clusters_patrimoniais_por_candidato.parquet",
        index=False
    )
except Exception as e:
    print("Parquet auxiliar não exportado:", e)

# Resumo dos perfis finais
resumo_perfis_finais.to_csv(
    "resumo_perfis_patrimoniais_finais.csv",
    index=False,
    sep=";"
)

# Auditoria dos nomes automáticos dos clusters
nomes_clusters_automaticos.to_csv(
    "nomes_clusters_automaticos.csv",
    index=False,
    sep=";"
)

print("Arquivos exportados:")
print("- features_candidatos_com_clusters_patrimoniais.csv")
print("- clusters_patrimoniais_por_candidato.csv")
print("- resumo_perfis_patrimoniais_finais.csv")
print("- nomes_clusters_automaticos.csv")


Arquivos exportados:
- features_candidatos_com_clusters_patrimoniais.csv
- clusters_patrimoniais_por_candidato.csv
- resumo_perfis_patrimoniais_finais.csv
- nomes_clusters_automaticos.csv


## 15. Opcional: aplicar clusters a um dataset original em outro nível de granularidade

Se houver um dataset original em nível de bem declarado, use a tabela `clusters_por_candidato` para aplicar a classificação final a todas as linhas do candidato.

Mantenha esta seção comentada se o notebook atual já estiver usando o dataset de features por candidato.


In [43]:
# Exemplo de uso, caso exista um dataframe original chamado df_original:
#
# df_original_com_clusters = df_original.merge(
#     clusters_por_candidato[[
#         "SQ_CANDIDATO",
#         "estrato_final",
#         "cluster_final_codigo",
#         "perfil_patrimonial_final"
#     ]],
#     on="SQ_CANDIDATO",
#     how="left"
# )
#
# df_original_com_clusters.to_csv(
#     "dataset_original_com_clusters_patrimoniais.csv",
#     index=False,
#     sep=";"
# )


## 16. Opcional: testes de robustez ainda dentro de clustering

As células abaixo não fazem análise complementar externa. Elas servem apenas para avaliar estabilidade/robustez da solução de clustering.

Por padrão, não são necessárias para gerar o dataset final.


In [44]:
def testar_estabilidade_kmeans(X, k, sementes=range(10, 60, 5), n_init=N_INIT_KMEANS):
    labels_por_seed = {}

    for seed in sementes:
        modelo = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=n_init
        )
        labels_por_seed[seed] = modelo.fit_predict(X)

    resultados = []
    seeds = list(labels_por_seed.keys())

    for i in range(len(seeds)):
        for j in range(i + 1, len(seeds)):
            resultados.append({
                "seed_1": seeds[i],
                "seed_2": seeds[j],
                "ari": adjusted_rand_score(
                    labels_por_seed[seeds[i]],
                    labels_por_seed[seeds[j]]
                )
            })

    return pd.DataFrame(resultados)


# Exemplo de uso:
# estabilidade_baixo = testar_estabilidade_kmeans(X_baixo_final, K_BAIXO)
# estabilidade_baixo["ari"].describe()


In [45]:
def testar_gmm(X, k_min, k_max):
    resultados = []

    for k in range(k_min, k_max + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type="diag",
            random_state=RANDOM_STATE,
            n_init=10
        )

        labels = gmm.fit_predict(X)
        contagem = pd.Series(labels).value_counts()

        resultados.append({
            "k": k,
            "bic": gmm.bic(X),
            "aic": gmm.aic(X),
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "menor_cluster": contagem.min(),
            "maior_cluster": contagem.max(),
            "perc_maior_cluster": contagem.max() / len(X)
        })

    return pd.DataFrame(resultados)


# Exemplo de uso:
# avaliacao_gmm_intermediario = testar_gmm(X_intermediario_final, 6, 12)
# avaliacao_gmm_intermediario


In [46]:
# Distância entre perfis médios finais para verificar possível redundância
variaveis_perfil_redundancia = [
    "perc_imoveis",
    "perc_veiculos",
    "perc_ativos_financeiros",
    "perc_participacoes_societarias",
    "perc_rural_agropecuario",
    "perc_creditos_direitos",
    "perc_dinheiro_especie",
    "perc_outros",
    "log_patrimonio_total",
    "log_qtd_bens",
    "qtd_macros_presentes",
    "indice_concentracao_macro"
]

perfil_medio_final = (
    features_candidatos
    .groupby("perfil_patrimonial_final")[variaveis_perfil_redundancia]
    .mean()
)

dist_perfis = pd.DataFrame(
    squareform(pdist(perfil_medio_final, metric="euclidean")),
    index=perfil_medio_final.index,
    columns=perfil_medio_final.index
)

pares = []
for i, p1 in enumerate(dist_perfis.index):
    for j, p2 in enumerate(dist_perfis.columns):
        if j > i:
            pares.append({
                "perfil_1": p1,
                "perfil_2": p2,
                "distancia": dist_perfis.loc[p1, p2]
            })

pares_proximos = pd.DataFrame(pares).sort_values("distancia")
pares_proximos.head(20)


,perfil_1,perfil_2,distancia
65,Alto patrimônio - imobiliário financeiro,Alto patrimônio - imobiliário financeiro crédi...,0.370203
495,Patrimônio intermediário - imobiliário misto,Patrimônio intermediário - imobiliário rural a...,0.392338
66,Alto patrimônio - imobiliário financeiro,Alto patrimônio - imobiliário rural agropecuário,0.409134
429,Patrimônio intermediário - créditos direitos m...,Patrimônio intermediário - imobiliário misto,0.441548
430,Patrimônio intermediário - créditos direitos m...,Patrimônio intermediário - imobiliário rural a...,0.512320
128,Alto patrimônio - imobiliário rural agropecuário,Alto patrimônio - imobiliário societário,0.539218
483,Patrimônio intermediário - imobiliário financeiro,Patrimônio intermediário - imobiliário misto,0.561775
96,Alto patrimônio - imobiliário financeiro crédi...,Alto patrimônio - imobiliário rural agropecuário,0.592396
484,Patrimônio intermediário - imobiliário financeiro,Patrimônio intermediário - imobiliário rural a...,0.631087
155,Alto patrimônio - imobiliário rural agropecuár...,Alto patrimônio - imobiliário rural agropecuár...,0.639241
